# 03 · Líneas base y modelos candidatos

Todos los modelos usan **la misma preparación** (`crear_pipeline`: `PerfilCantonal` + `ColumnTransformer`), **la misma validación** (`SplitTemporal`: 5 pliegues de ventana creciente sobre fechas, hueco de 3 días, P90 del target recalculado en cada pliegue) y **las mismas métricas**:

- **Técnica — PR-AUC** (average precision), definida en el PSet 1. Con 8% de positivos, accuracy o ROC-AUC se ven bien aunque el modelo no encuentre los picos; PR-AUC mide precisión y recall de la clase rara. Referencia: un modelo sin información obtiene PR-AUC = prevalencia.
- **De negocio — reducción de días de alta demanda no anticipados (FN) frente a la regla estacional, con el mismo número de alertas.** El error más costoso del proyecto es no anticipar un pico (costo FN:FP = 5:1). Al darle al modelo exactamente tantas alertas como emitió la regla, el costo de revisión es igual y solo cambia cuántos picos se dejan pasar. Meta del PSet 1: ≥ 10%.

Cada corrida de MLflow registra hiperparámetros y semilla, datos (tabla de origen, fecha de corte del snapshot, hash), métricas de entrenamiento y de CV (media, desviación y por pliegue), segundos de entrenamiento y de predicción, tags (`fase`, `familia`, `git_commit`) y el pipeline reentrenado con todo el entrenamiento (`mlflow.sklearn.log_model` con firma y ejemplo de entrada). El conjunto de prueba **no se usa** en este notebook.

In [ ]:
import sys, warnings
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
FIG = RAIZ / "docs" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)

def guardar(fig, nombre):
    fig.savefig(FIG / f"{nombre}.png", dpi=130, bbox_inches="tight")

In [ ]:
import time
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

from src.datos import cargar_obt, construir_dataset, dividir_train_test
from src.pipeline import (ENTRADA, SEMILLA, EXPERIMENTO, crear_pipeline, crear_regla_estacional,
                          crear_chronos, evaluar_cv, registrar_corrida, configurar_mlflow, contexto_git)

obt, meta = cargar_obt()
train, test = dividir_train_test(construir_dataset(obt))
del test                                   # el hold-out no se toca hasta el notebook 05
mlflow = configurar_mlflow()
exp = mlflow.get_experiment_by_name(EXPERIMENTO)
print(f"MLflow: {mlflow.get_tracking_uri()} | experimento '{exp.name}' (id {exp.experiment_id})")
print("Git:", contexto_git())

In [ ]:
resultados = []

def correr(nombre, pipe, fase, familia, **kw):
    """Evalúa con la CV temporal, registra la corrida en MLflow y guarda el resumen."""
    t0 = time.perf_counter()
    r = evaluar_cv(pipe, train, columnas=kw.get("columnas"), muestra_train=kw.get("muestra_train"))
    run_id, _ = registrar_corrida(nombre, pipe, train, meta, fase=fase, familia=familia, resultado=r, **kw)
    resultados.append({"modelo": nombre, "fase": fase, "familia": familia, "run_id": run_id,
                       **r.resumen, "pr_auc_por_pliegue": r.pliegues["pr_auc_val"].round(4).tolist()})
    print(f"{nombre:<26} PR-AUC CV {r.resumen['cv_pr_auc_mean']:.4f} ± {r.resumen['cv_pr_auc_std']:.4f} | "
          f"train {r.resumen['train_pr_auc']:.4f} | red. FN {r.resumen['cv_reduccion_fn_mean']:+.3f} | "
          f"{time.perf_counter() - t0:,.0f} s en total | run {run_id[:8]}")
    return r

## 1. Líneas base

1. **Ingenua:** `DummyClassifier(strategy="prior")` — siempre predice la prevalencia del entrenamiento (su PR-AUC es la prevalencia de cada pliegue).
2. **De negocio (PSet 1):** regla estacional — alerta si el promedio de las últimas 4 ocurrencias del mismo día de la semana supera el P90 del cantón.

In [ ]:
correr("dummy_prior", crear_pipeline(DummyClassifier(strategy="prior")), "baseline", "ingenua")
correr("regla_estacional", crear_regla_estacional(), "baseline", "regla_negocio");

## 2. Candidatos con hiperparámetros por defecto

| Familia | Modelo |
|---|---|
| Lineal regularizado | `LogisticRegression` (L2, C = 1) |
| Bagging | `RandomForestClassifier` |
| Boosting | `HistGradientBoostingClassifier` |
| Red neuronal | `MLPClassifier` |
| State of the Art | Chronos-Bolt (sección 3) |

Solo se fija la semilla (`random_state`) y `n_jobs=-1` en el Random Forest (paralelismo, no cambia el modelo).

In [ ]:
candidatos = {
    "logistic_regression": ("lineal", LogisticRegression(random_state=SEMILLA)),
    "random_forest": ("bagging", RandomForestClassifier(random_state=SEMILLA, n_jobs=-1)),
    "hist_gradient_boosting": ("boosting", HistGradientBoostingClassifier(random_state=SEMILLA)),
    "mlp": ("red_neuronal", MLPClassifier(random_state=SEMILLA)),
}
for nombre, (familia, est) in candidatos.items():
    correr(nombre, crear_pipeline(est), "candidato", familia)

## 3. State of the Art: Chronos-Bolt (modelo fundacional, zero-shot)

`amazon/chronos-bolt-small` es un modelo fundacional preentrenado en miles de series de tiempo. No se entrena con nuestros datos: para cada fila (cantón, *t*) recibe la serie diaria del cantón hasta *t* (400 días de contexto), pronostica los cuantiles 0,1…0,9 de *t+3* y `prob_superar` los convierte en P(volumen > P90 del cantón). Usa la **misma CV, las mismas etiquetas y la misma métrica**; lo único distinto es la entrada (la serie en lugar de las features), porque así funciona el modelo. Como predecir es costoso (~2 min por cada 10.000 filas en CPU), la métrica de entrenamiento se estima sobre 20.000 filas de cada pliegue; la de validación usa los pliegues completos.

In [ ]:
correr("chronos_bolt_small", crear_chronos(), "candidato", "state_of_the_art",
       columnas=ENTRADA + ["fecha"], muestra_train=20_000,
       params={"sota_contexto_dias": 400, "sota_muestra_train": 20_000});

**LLM mediante prompting:** pendiente para una versión posterior (fuera del alcance de esta entrega parcial).

## 4. Resultados de todos los modelos

In [ ]:
res = pd.DataFrame(resultados)
lr = {"hist_gradient_boosting": 0.1, "mlp": 0.001}     # learning rate por defecto (HGB: learning_rate; MLP: learning_rate_init)
tabla = pd.DataFrame({
    "modelo": res["modelo"], "fase": res["fase"], "familia": res["familia"],
    "pr_auc_train": res["train_pr_auc"],
    "pr_auc_cv": res["cv_pr_auc_mean"].map("{:.4f}".format) + " ± " + res["cv_pr_auc_std"].map("{:.4f}".format),
    "brecha_train_cv": res["brecha_pr_auc"],
    "reduccion_fn_cv": res["cv_reduccion_fn_mean"].map("{:+.1%}".format) + " ± " + res["cv_reduccion_fn_std"].map("{:.1%}".format),
    "learning_rate": res["modelo"].map(lr),
    "seg_fit_por_pliegue": res["seg_fit_promedio"], "seg_pred_por_pliegue": res["seg_pred_promedio"],
    "run_id": res["run_id"],
})
tabla.to_csv(RAIZ / "docs" / "resultados_candidatos.csv", index=False)
tabla

In [ ]:
orden = res.sort_values("cv_pr_auc_mean")
colores = orden["fase"].map({"baseline": "#999999", "candidato": "#4c72b0"})
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
ax[0].barh(orden["modelo"], orden["cv_pr_auc_mean"], xerr=orden["cv_pr_auc_std"], color=colores, label="CV (media ± desv.)")
ax[0].scatter(orden["train_pr_auc"], orden["modelo"], color="red", marker="|", s=300, label="entrenamiento")
ax[0].set(xlabel="PR-AUC", title="PR-AUC: entrenamiento vs CV"); ax[0].legend(loc="lower right")
ax[1].barh(orden["modelo"], orden["brecha_pr_auc"], color=colores)
ax[1].set(xlabel="PR-AUC train − PR-AUC CV", title="Brecha (sobreajuste)")
ax[2].barh(orden["modelo"], orden["seg_fit_promedio"] + orden["seg_pred_promedio"], color=colores)
ax[2].set(xscale="log", xlabel="segundos por pliegue (ajuste + predicción, escala log)", title="Costo")
fig.tight_layout(); guardar(fig, "03_candidatos_resumen")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for _, f in res.iterrows():
    ax.plot(range(1, 6), f["pr_auc_por_pliegue"], marker="o", label=f["modelo"],
            ls="--" if f["fase"] == "baseline" else "-")
ax.set(xticks=range(1, 6), xlabel="pliegue (más reciente →)", ylabel="PR-AUC validación",
       title="PR-AUC por pliegue: el orden entre modelos se mantiene en cada periodo")
ax.legend(fontsize=8, ncol=2); fig.tight_layout(); guardar(fig, "03_candidatos_por_pliegue")

## 5. ¿Superan a las dos líneas base? y diagnóstico de sobreajuste

In [ ]:
base = res.set_index("modelo")
dummy, regla = base.loc["dummy_prior"], base.loc["regla_estacional"]
cand = res[res["fase"] == "candidato"].copy()
cand["supera_dummy"] = cand["cv_pr_auc_mean"] > dummy["cv_pr_auc_mean"] + dummy["cv_pr_auc_std"]
cand["supera_regla"] = cand["cv_pr_auc_mean"] > regla["cv_pr_auc_mean"] + regla["cv_pr_auc_std"]
cand["brecha_relativa"] = cand["brecha_pr_auc"] / cand["cv_pr_auc_mean"]
cand["diagnostico"] = np.select(
    [cand["brecha_relativa"] > 1, cand["brecha_relativa"] > 0.25],
    ["sobreajuste fuerte (memoriza el entrenamiento)", "sobreajuste moderado"], "sin sobreajuste relevante")
cand[["modelo", "cv_pr_auc_mean", "cv_pr_auc_std", "train_pr_auc", "brecha_pr_auc", "brecha_relativa",
      "supera_dummy", "supera_regla", "cv_reduccion_fn_mean", "diagnostico"]].sort_values("cv_pr_auc_mean", ascending=False)

## 6. Los 3 mejores candidatos pasan al ajuste de hiperparámetros

Se eligen por PR-AUC medio en CV. Chronos-Bolt es zero-shot (no tiene hiperparámetros de entrenamiento) y evaluar una configuración cuesta ~1 hora de CPU, así que no es candidato a 30 configuraciones de tuning aunque quede entre los mejores; se reporta como referencia.

In [ ]:
ajustables = cand[cand["modelo"] != "chronos_bolt_small"].sort_values("cv_pr_auc_mean", ascending=False)
top3 = ajustables.head(3)[["modelo", "familia", "run_id", "cv_pr_auc_mean", "cv_pr_auc_std"]]
top3.to_csv(RAIZ / "docs" / "top3_para_tuning.csv", index=False)
top3